# EDA: Monitoring Time-Series Data

Exploratory data analysis for the ABPM hemodynamic coupling study.  
This notebook covers univariate distributions, context labels, missingness,
outliers, circadian patterns, between-subject variability, and correlations.

In [ ]:
import sys
import os

# Ensure CWD is project root so Config relative paths resolve correctly
os.chdir(os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))
sys.path.insert(0, ".")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy import stats

from abpm_hemodynamic_coupling.config import Config, Columns
from abpm_hemodynamic_coupling.data_processing import DataLoader
from analysis.thesis_stats import (
    bootstrap_ci, format_median_iqr, variance_decomposition,
    compute_icc1, add_sin_cos_hour, export_table
)

config = Config()
THESIS_DIR = Path("results/thesis")
THESIS_DIR.mkdir(parents=True, exist_ok=True)
DPI = 300

sns.set_theme(style="whitegrid", font_scale=1.1)
pd.set_option("display.max_columns", 30)

In [ ]:
# Load monitoring data directly (bypassing strict validator to preserve all raw values for EDA)
df = pd.read_csv(config.get_data_path(config.MONITORING_FILE))
df[Columns.TIME] = pd.to_datetime(df[Columns.TIME])
df = df.sort_values([Columns.PAT_ID, Columns.TIME])
df = df.dropna(subset=[Columns.SBP, Columns.DBP, Columns.HR])

# Apply hierarchical labeling
from abpm_hemodynamic_coupling.data_processing import Labeler
df[Columns.LABEL] = df.apply(Labeler.apply_hierarchy, axis=1)

print(f"Loaded: {len(df)} readings, {df[Columns.PAT_ID].nunique()} subjects")
print(f"\nValue ranges:")
for var in [Columns.SBP, Columns.DBP, Columns.HR]:
    print(f"  {var}: {df[var].min():.0f} – {df[var].max():.0f}")
print(f"\nLabel distribution:\n{df[Columns.LABEL].value_counts()}")

## 1. Univariate Distributions (SBP, DBP, HR)

In [ ]:
# Pooled histograms + KDE for SBP, DBP, HR
hemodynamic_vars = [Columns.SBP, Columns.DBP, Columns.HR]
units = {Columns.SBP: "mmHg", Columns.DBP: "mmHg", Columns.HR: "bpm"}
palette = sns.color_palette("Set2")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for ax, var, color in zip(axes, hemodynamic_vars, palette):
    vals = df[var].dropna()
    ax.hist(vals, bins="fd", density=True, alpha=0.6, color=color, edgecolor="white")
    vals.plot.kde(ax=ax, color="black", linewidth=1.5)
    med = vals.median()
    ax.axvline(med, color="red", linestyle="--", linewidth=1.2, label=f"Median = {med:.1f}")
    summary = format_median_iqr(vals)
    ax.text(
        0.97, 0.95, f"Mdn (IQR)\n{summary}",
        transform=ax.transAxes, ha="right", va="top",
        fontsize=9, bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8),
    )
    ax.set_xlabel(f"{var} ({units[var]})")
    ax.set_ylabel("Density")
    ax.set_title(var)
    ax.legend(fontsize=8)

plt.tight_layout()
fig.savefig(THESIS_DIR / "fig_01_hemodynamic_distributions.png", dpi=DPI, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# Subject-faceted mini-histograms for SBP
g = sns.FacetGrid(
    df, col=Columns.PAT_ID, col_wrap=7, height=2, aspect=1.2,
    sharex=True, sharey=True,
)
g.map(plt.hist, Columns.SBP, bins=15, color=palette[0], edgecolor="white", alpha=0.7)
g.set_titles("ID {col_name}")
g.set_xlabels("SBP (mmHg)")
g.set_ylabels("Count")
g.tight_layout()
plt.show()
plt.close()

In [ ]:
# Per-subject summary violin plots
subject_medians = df.groupby(Columns.PAT_ID)[hemodynamic_vars].median().reset_index()

fig, axes = plt.subplots(1, 3, figsize=(14, 5))

for ax, var, color in zip(axes, hemodynamic_vars, palette):
    sns.violinplot(y=subject_medians[var], ax=ax, color=color, inner=None, alpha=0.4)
    sns.stripplot(y=subject_medians[var], ax=ax, color="black", size=4, jitter=0.15, alpha=0.7)
    ax.set_ylabel(f"Subject median {var} ({units[var]})")
    ax.set_title(var)

plt.tight_layout()
plt.show()
plt.close(fig)

## 2. Context Label Distribution

In [ ]:
# Label frequency barplots: (a) reading counts, (b) subject counts
label_counts = df[Columns.LABEL].value_counts().sort_values()
label_subject_counts = df.groupby(Columns.LABEL)[Columns.PAT_ID].nunique().sort_values()
total_readings = len(df)
total_subjects = df[Columns.PAT_ID].nunique()

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# (a) Reading counts
bars1 = ax1.barh(label_counts.index, label_counts.values, color=palette[0])
for bar, count in zip(bars1, label_counts.values):
    pct = count / total_readings * 100
    ax1.text(bar.get_width() + total_readings * 0.01, bar.get_y() + bar.get_height() / 2,
             f"{count} ({pct:.1f}%)", va="center", fontsize=8)
ax1.set_xlabel("Number of readings")
ax1.set_title("(a) Readings per context label")

# (b) Subject counts
bars2 = ax2.barh(label_subject_counts.index, label_subject_counts.values, color=palette[1])
for bar, count in zip(bars2, label_subject_counts.values):
    pct = count / total_subjects * 100
    ax2.text(bar.get_width() + total_subjects * 0.01, bar.get_y() + bar.get_height() / 2,
             f"{count} ({pct:.1f}%)", va="center", fontsize=8)
ax2.set_xlabel("Number of subjects")
ax2.set_title("(b) Subjects per context label")

plt.tight_layout()
plt.show()
plt.close(fig)

## 3. Missing Data Analysis

In [ ]:
# Missingness per column
miss_pct = df.isna().mean() * 100
miss_pct = miss_pct[miss_pct.index.isin(hemodynamic_vars + [Columns.TIME, Columns.STATE, Columns.LABEL])]
miss_pct = miss_pct.sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(miss_pct.index, miss_pct.values, color=palette[3])
for bar, pct in zip(bars, miss_pct.values):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height() / 2,
            f"{pct:.2f}%", va="center", fontsize=9)
ax.set_xlabel("Missing (%)")
ax.set_title("Missingness per key column")
plt.tight_layout()
plt.show()
plt.close(fig)

print(f"Overall missingness in hemodynamic columns: {df[hemodynamic_vars].isna().mean().mean() * 100:.2f}%")

In [ ]:
# Readings per subject
readings_per_subj = df.groupby(Columns.PAT_ID).size().sort_values()
median_count = readings_per_subj.median()
colors = [palette[0] if c >= median_count else palette[3] for c in readings_per_subj.values]

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(readings_per_subj.index.astype(str), readings_per_subj.values, color=colors)
ax.axhline(median_count, color="red", linestyle="--", linewidth=1, label=f"Median = {median_count:.0f}")
ax.set_xlabel("Participant ID")
ax.set_ylabel("Number of readings")
ax.set_title("Data availability per subject")
ax.legend()
plt.xticks(rotation=90, fontsize=7)
plt.tight_layout()
plt.show()
plt.close(fig)

print(f"Readings per subject: {format_median_iqr(readings_per_subj.values)}")

## 4. Outlier Detection

In [ ]:
# Clinical outlier flags
df["outlier_SBP"] = (df[Columns.SBP] < 70) | (df[Columns.SBP] > 180)
df["outlier_DBP"] = (df[Columns.DBP] < 40) | (df[Columns.DBP] > 120)
df["outlier_HR"] = (df[Columns.HR] < 45) | (df[Columns.HR] > 150)
df["any_outlier"] = df["outlier_SBP"] | df["outlier_DBP"] | df["outlier_HR"]

outlier_summary = pd.DataFrame({
    "Variable": ["SBP (<70 or >180)", "DBP (<40 or >120)", "HR (<45 or >150)", "Any"],
    "Count": [
        df["outlier_SBP"].sum(),
        df["outlier_DBP"].sum(),
        df["outlier_HR"].sum(),
        df["any_outlier"].sum(),
    ],
    "Percentage": [
        df["outlier_SBP"].mean() * 100,
        df["outlier_DBP"].mean() * 100,
        df["outlier_HR"].mean() * 100,
        df["any_outlier"].mean() * 100,
    ],
})
print(outlier_summary.to_string(index=False))

# Strip plots with outliers highlighted
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, var, outlier_col, color in zip(
    axes, hemodynamic_vars,
    ["outlier_SBP", "outlier_DBP", "outlier_HR"],
    palette,
):
    normal = df[~df[outlier_col]]
    outliers = df[df[outlier_col]]
    sns.stripplot(
        data=normal, x=Columns.PAT_ID, y=var, ax=ax,
        color=color, size=2, alpha=0.3, jitter=0.3,
    )
    if len(outliers) > 0:
        sns.stripplot(
            data=outliers, x=Columns.PAT_ID, y=var, ax=ax,
            color="red", size=4, alpha=0.9, jitter=0.3, marker="X",
        )
    ax.set_title(f"{var} by subject (outliers in red)")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=90, labelsize=6)

plt.tight_layout()
plt.show()
plt.close(fig)

In [ ]:
# Outliers by context label
outlier_by_label = df.groupby(Columns.LABEL)[
    ["outlier_SBP", "outlier_DBP", "outlier_HR"]
].sum().sort_values("outlier_SBP", ascending=False)

outlier_by_label.columns = ["SBP", "DBP", "HR"]

fig, ax = plt.subplots(figsize=(10, 5))
outlier_by_label.plot(kind="barh", ax=ax, color=palette[:3])
ax.set_xlabel("Number of outlier readings")
ax.set_title("Clinical outlier counts by context label")
ax.legend(title="Variable")
plt.tight_layout()
plt.show()
plt.close(fig)

## 5. Temporal / Circadian Patterns

In [ ]:
# Circadian profiles: hourly median + IQR ribbon
df["hour"] = df[Columns.TIME].dt.hour

hourly_stats = df.groupby("hour")[hemodynamic_vars].agg(["median", lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)])
hourly_stats.columns = ["_".join(col).strip() for col in hourly_stats.columns]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, var, color in zip(axes, hemodynamic_vars, palette):
    med_col = f"{var}_median"
    q25_col = f"{var}_<lambda_0>"
    q75_col = f"{var}_<lambda_1>"
    hours = hourly_stats.index
    ax.fill_between(
        hours, hourly_stats[q25_col], hourly_stats[q75_col],
        alpha=0.3, color=color,
    )
    ax.plot(hours, hourly_stats[med_col], color=color, linewidth=2, marker="o", markersize=4)
    # Shade sleep hours (22-07)
    ax.axvspan(0, 7, alpha=0.08, color="navy", label="Typical sleep")
    ax.axvspan(22, 24, alpha=0.08, color="navy")
    ax.set_xlabel("Hour of day")
    ax.set_ylabel(f"{var} ({units[var]})")
    ax.set_title(f"{var} circadian profile")
    ax.set_xticks(range(0, 24, 3))
    ax.legend(fontsize=8, loc="upper right")

plt.tight_layout()
fig.savefig(THESIS_DIR / "fig_02_circadian_profiles.png", dpi=DPI, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# Day vs Night violin comparison (state: 0=wake, 1=sleep)
df["period"] = df[Columns.STATE].map({0: "Wake", 1: "Sleep"})
df_period = df.dropna(subset=["period"])

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for ax, var, color in zip(axes, hemodynamic_vars, palette):
    sns.violinplot(
        data=df_period, x="period", y=var, ax=ax,
        palette=[palette[0], palette[1]], inner="quartile", alpha=0.6,
    )
    # Paired dot plot: subject medians linked by lines
    subj_period_med = df_period.groupby([Columns.PAT_ID, "period"])[var].median().reset_index()
    subj_pivot = subj_period_med.pivot(index=Columns.PAT_ID, columns="period", values=var).dropna()
    for _, row in subj_pivot.iterrows():
        ax.plot([0, 1], [row["Wake"], row["Sleep"]], color="gray", alpha=0.3, linewidth=0.7)
    ax.set_title(f"{var}: Wake vs Sleep")
    ax.set_xlabel("")
    ax.set_ylabel(f"{var} ({units[var]})")

plt.tight_layout()
fig.savefig(THESIS_DIR / "fig_03_sleep_transition_profiles.png", dpi=DPI, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# Within-subject variability by hour
hourly_subj_sd = df.groupby(["hour", Columns.PAT_ID])[hemodynamic_vars].std().reset_index()
hourly_sd_stats = hourly_subj_sd.groupby("hour")[hemodynamic_vars].agg(["median", lambda x: x.quantile(0.25), lambda x: x.quantile(0.75)])
hourly_sd_stats.columns = ["_".join(col).strip() for col in hourly_sd_stats.columns]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, var, color in zip(axes, hemodynamic_vars, palette):
    med_col = f"{var}_median"
    q25_col = f"{var}_<lambda_0>"
    q75_col = f"{var}_<lambda_1>"
    hours = hourly_sd_stats.index
    ax.fill_between(
        hours, hourly_sd_stats[q25_col], hourly_sd_stats[q75_col],
        alpha=0.3, color=color,
    )
    ax.plot(hours, hourly_sd_stats[med_col], color=color, linewidth=2, marker="o", markersize=4)
    ax.axvspan(0, 7, alpha=0.08, color="navy")
    ax.axvspan(22, 24, alpha=0.08, color="navy")
    ax.set_xlabel("Hour of day")
    ax.set_ylabel(f"Within-subject SD of {var}")
    ax.set_title(f"{var} variability by hour")
    ax.set_xticks(range(0, 24, 3))

plt.tight_layout()
plt.show()
plt.close(fig)

## 6. Between-Subject Variability

In [ ]:
# Spaghetti plots: per-subject hourly median trajectories
subj_hourly = df.groupby([Columns.PAT_ID, "hour"])[hemodynamic_vars].median().reset_index()
cohort_hourly = df.groupby("hour")[hemodynamic_vars].median()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, var, color in zip(axes, hemodynamic_vars, palette):
    for pid, grp in subj_hourly.groupby(Columns.PAT_ID):
        ax.plot(grp["hour"], grp[var], color="gray", alpha=0.25, linewidth=0.7)
    ax.plot(
        cohort_hourly.index, cohort_hourly[var],
        color=color, linewidth=2.5, label="Cohort median",
    )
    ax.set_xlabel("Hour of day")
    ax.set_ylabel(f"{var} ({units[var]})")
    ax.set_title(f"{var} subject trajectories")
    ax.set_xticks(range(0, 24, 3))
    ax.legend(fontsize=8)

plt.tight_layout()
fig.savefig(THESIS_DIR / "fig_04_spaghetti_subject_trajectories.png", dpi=DPI, bbox_inches="tight")
plt.show()
plt.close(fig)

In [ ]:
# Variance decomposition & ICC
var_decomp_rows = []
for var in hemodynamic_vars:
    vd = variance_decomposition(df, Columns.PAT_ID, var)
    icc_val, icc_lo, icc_hi = compute_icc1(df, Columns.PAT_ID, var)
    var_decomp_rows.append({
        "Variable": var,
        "Between-subject var": round(vd["between_var"], 2),
        "Within-subject var": round(vd["within_var"], 2),
        "Total var": round(vd["total_var"], 2),
        "Between %": round(vd["between_var"] / vd["total_var"] * 100, 1) if vd["total_var"] > 0 else 0.0,
        "ICC(1)": round(icc_val, 3),
        "ICC CI low": round(icc_lo, 3) if icc_lo is not None else None,
        "ICC CI high": round(icc_hi, 3) if icc_hi is not None else None,
    })

var_decomp_df = pd.DataFrame(var_decomp_rows)
print(var_decomp_df.to_string(index=False))

export_table(var_decomp_df, THESIS_DIR / "table_03_variance_decomposition_icc")
print(f"\nSaved to {THESIS_DIR / 'table_03_variance_decomposition_icc.csv'}")

## 7. Correlation Analysis (Long-Format)

In [ ]:
# Pooled Spearman correlation heatmap
corr_matrix = df[hemodynamic_vars].corr(method="spearman")

fig, ax = plt.subplots(figsize=(5, 4))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(
    corr_matrix, annot=True, fmt=".2f", cmap="coolwarm",
    mask=mask, vmin=-1, vmax=1, center=0,
    square=True, linewidths=1, ax=ax,
)
ax.set_title("Pooled Spearman correlations")
plt.tight_layout()
plt.show()
plt.close(fig)

In [ ]:
# Per-subject coupling analysis
pairs = [(Columns.SBP, Columns.DBP), (Columns.SBP, Columns.HR), (Columns.DBP, Columns.HR)]
pair_labels = ["SBP-DBP", "SBP-HR", "DBP-HR"]

corr_records = []
for pid, grp in df.groupby(Columns.PAT_ID):
    for (v1, v2), label in zip(pairs, pair_labels):
        if len(grp) >= 5:
            rho, p = stats.spearmanr(grp[v1], grp[v2])
            corr_records.append({Columns.PAT_ID: pid, "pair": label, "rho": rho, "p": p})

corr_df = pd.DataFrame(corr_records)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# (a) Heatmap: participant x pair
corr_pivot = corr_df.pivot(index=Columns.PAT_ID, columns="pair", values="rho")
sns.heatmap(
    corr_pivot, annot=True, fmt=".2f", cmap="coolwarm",
    vmin=-1, vmax=1, center=0, linewidths=0.5, ax=ax1,
)
ax1.set_title("Per-subject Spearman rho")
ax1.set_ylabel("Participant")

# (b) Distribution of rho values
sns.boxplot(data=corr_df, x="pair", y="rho", ax=ax2, palette="Set2", width=0.5)
sns.stripplot(data=corr_df, x="pair", y="rho", ax=ax2, color="black", size=3, alpha=0.5)
ax2.axhline(0, color="gray", linestyle="--", linewidth=0.8)
ax2.set_ylabel("Spearman rho")
ax2.set_xlabel("")
ax2.set_title("Distribution of per-subject correlations")

plt.tight_layout()
fig.savefig(THESIS_DIR / "fig_05_subject_correlation_heatmaps.png", dpi=DPI, bbox_inches="tight")
plt.show()
plt.close(fig)

# Summary
for label in pair_labels:
    subset = corr_df[corr_df["pair"] == label]["rho"]
    print(f"{label}: {format_median_iqr(subset.values)}")

## 8. Save Labeled Dataset

In [ ]:
# Export labeled monitoring data and subject-context medians
cols_to_export = [c for c in df.columns if c not in ["outlier_SBP", "outlier_DBP", "outlier_HR", "any_outlier", "hour", "period"]]
df[cols_to_export].to_parquet(THESIS_DIR / "monitoring_labeled.parquet", index=False)

subject_context = df.groupby([Columns.PAT_ID, Columns.LABEL]).agg(
    SBP_med=(Columns.SBP, "median"),
    DBP_med=(Columns.DBP, "median"),
    HR_med=(Columns.HR, "median"),
    SBP_iqr=(Columns.SBP, lambda x: x.quantile(0.75) - x.quantile(0.25)),
    DBP_iqr=(Columns.DBP, lambda x: x.quantile(0.75) - x.quantile(0.25)),
    HR_iqr=(Columns.HR, lambda x: x.quantile(0.75) - x.quantile(0.25)),
    n=(Columns.SBP, "count"),
).reset_index()
subject_context.to_csv(THESIS_DIR / "subject_context_medians.csv", index=False)
print(f"Exported: {len(subject_context)} rows")
print(f"Parquet: {THESIS_DIR / 'monitoring_labeled.parquet'}")
print(f"CSV:     {THESIS_DIR / 'subject_context_medians.csv'}")